<a href="https://colab.research.google.com/github/aastha3399/ML_Lab_2026/blob/main/ML_EXP11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

class PCA:
    """
    Principal Component Analysis (PCA) implementation from scratch using NumPy.

    Parameters
    ----------
    n_components : int or None, default=None
        Number of principal components to keep. If None, all components are retained.
    """

    def __init__(self, n_components=None):
        self.n_components = n_components
        self.components_ = None  # Eigenvectors (shape: [n_components, n_features])
        self.explained_variance_ = None  # Eigenvalues (variances of components)
        self.explained_variance_ratio_ = None  # Ratio of total variance explained
        self.mean_ = None  # Empirical feature means

    def fit(self, X):
        """
        Fit the model with matrix X.

        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)
        """
        X = np.asarray(X, dtype=np.float64)
        n_samples, n_features = X.shape

        if self.n_components is not None and self.n_components > min(
            n_samples, n_features
        ):
            raise ValueError(
                f"n_components={self.n_components} cannot exceed min(n_samples, n_features)="
                f"{min(n_samples, n_features)}"
            )

        # 1. Center the data (zero-mean per feature)
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_

        # 2. Compute sample covariance matrix: Sigma = (1 / (N - 1)) * (X_c.T @ X_c)
        cov_matrix = np.cov(X_centered, rowvar=False)

        # 3. Eigendecomposition of the symmetric covariance matrix
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

        # 4. Sort eigenvalues and eigenvectors in descending order
        sorted_indices = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[sorted_indices]
        eigenvectors = eigenvectors[:, sorted_indices]

        # 5. Calculate total variance and explained variance ratio
        total_variance = np.sum(eigenvalues)
        self.explained_variance_ = eigenvalues
        self.explained_variance_ratio_ = (
            eigenvalues / total_variance if total_variance > 0 else eigenvalues
        )

        # 6. Select top n_components
        k = self.n_components if self.n_components is not None else n_features
        # Store eigenvectors as rows: shape (k, n_features)
        self.components_ = eigenvectors[:, :k].T
        self.explained_variance_ = self.explained_variance_[:k]
        self.explained_variance_ratio_ = self.explained_variance_ratio_[:k]

        return self

    def transform(self, X):
        """
        Project X into the principal component subspace.

        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)

        Returns
        -------
        X_projected : ndarray of shape (n_samples, n_components)
        """
        X = np.asarray(X, dtype=np.float64)
        X_centered = X - self.mean_
        return np.dot(X_centered, self.components_.T)

    def fit_transform(self, X):
        """Fit the model with X and project X."""
        return self.fit(X).transform(X)

    def inverse_transform(self, X_projected):
        """
        Reconstruct data back to the original feature space.

        Parameters
        ----------
        X_projected : ndarray of shape (n_samples, n_components)

        Returns
        -------
        X_reconstructed : ndarray of shape (n_samples, n_features)
        """
        return np.dot(X_projected, self.components_) + self.mean_


# ---------------------------------------------------------
# Verification & Example Usage
# ---------------------------------------------------------
if __name__ == "__main__":
    # Generate synthetic correlated 2D dataset: Y ≈ 2*X + noise
    np.random.seed(42)
    x1 = np.random.normal(loc=10.0, scale=3.0, size=150)
    x2 = 2.0 * x1 + np.random.normal(loc=0.0, scale=1.0, size=150)
    data = np.column_stack([x1, x2])

    # Fit PCA and reduce to 2 components
    pca = PCA(n_components=2)
    transformed_data = pca.fit_transform(data)

    print("=== Empirical Feature Means ===")
    print(pca.mean_)

    print("\n=== Principal Components (Eigenvectors as rows) ===")
    print(pca.components_)

    print("\n=== Eigenvalues (Explained Variance) ===")
    print(pca.explained_variance_)

    print("\n=== Explained Variance Ratio ===")
    for idx, ratio in enumerate(pca.explained_variance_ratio_, 1):
        print(f"PC{idx}: {ratio * 100:.2f}%")

    # Verify reconstruction loss with 1 component
    pca_1d = PCA(n_components=1)
    data_1d = pca_1d.fit_transform(data)
    data_reconstructed = pca_1d.inverse_transform(data_1d)
    reconstruction_error = np.mean((data - data_reconstructed) ** 2)

    print(f"\nMean Squared Reconstruction Error (1D projection): {reconstruction_error:.4f}")

=== Empirical Feature Means ===
[ 9.7530194  19.57726855]

=== Principal Components (Eigenvectors as rows) ===
[[ 0.4394601   0.89826211]
 [-0.89826211  0.4394601 ]]

=== Eigenvalues (Explained Variance) ===
[40.52322421  0.20576288]

=== Explained Variance Ratio ===
PC1: 99.49%
PC2: 0.51%

Mean Squared Reconstruction Error (1D projection): 0.1022
